# Track B - FINISHING: epoch-22 validation + final eval + report (Kaggle, 2xT4)

The retrain kernel (`ssgokulaswamy/trackb-official-retrain`) resumed at epoch 16 and trained
epochs 17-22 on the **official DOTA v2.0 trainval** split. It hit the Kaggle 12h cap during the
epoch-22 validation, so its own cells 6/7 (final eval + report) never ran.

Per decision, **epochs 23 and 24 are intentionally SKIPPED**. This notebook:

1. mounts the retrain kernel's committed output via `kernel_sources`, picks the last clean
   checkpoint (expected `epoch_22.pth`), and copies it into this notebook's own output,
2. **re-tiles the official DOTA v2.0 VAL split (7,122 patches) HERE on Kaggle** and proves it,
3. runs the epoch-22 official validation (`DOTAMetric` dota/mAP; `test_evaluator == val_evaluator`)
   and the final extended evaluation via `scripts/run_track_eval.py`,
4. writes honest deliverables labelled `epoch_22` / `17-22`, with 23/24 explicitly marked skipped,
5. saves **every important file into `/kaggle/working/ada_trackB_official_finish`** (Kaggle output)
   so nothing is left to fate. These are pulled later and pushed to GitHub.

No dataset/checkpoint is stored on the local system. Model, loss, optimizer, LR schedule,
augmentations, sampler and seed are unchanged from the retrain.

`val_evaluator = test_evaluator = DOTAMetric(metric='mAP')` and `test_dataloader = val_dataloader`,
so the `test.py` pass reproduces exactly the epoch-22 in-training validation that was cut.


In [ ]:
#@title 1 - Environment, repo, CLEAN checkpoint discovery { display-mode: "form" }
import subprocess, sys, os, re, json
from pathlib import Path

BASE = Path('/tmp/ada')
REPO = BASE / 'ADA'
DATA = Path('/tmp/data')
VENV = Path('/tmp/mmrotate_venv')
WORK = Path('/kaggle/working/work_dirs/trackB_finish_ep22')
FIN  = Path('/kaggle/working/ada_trackB_official_finish')   # <-- all deliverables land here
COMMIT = 'a2ecd69d3d487c3f639c3cec329e0a737454370e'
SRC_SLUG = 'ssgokulaswamy/trackb-official-retrain'

os.environ['MPLBACKEND'] = 'Agg'

print(subprocess.run(['nvidia-smi', '--query-gpu=index,name,memory.total',
                      '--format=csv,noheader'], capture_output=True, text=True).stdout.strip())
print('system python', sys.version.split()[0])
print(subprocess.run(['df', '-BG', '/tmp', '/kaggle/working'], capture_output=True, text=True).stdout)

for p in (BASE, DATA, WORK, FIN, FIN / 'logs', FIN / 'checkpoints', FIN / 'val_annfiles'):
    p.mkdir(parents=True, exist_ok=True)

# config embedded in the checkpoints hard-codes /content/data/split_ss_dotav2
Path('/content').mkdir(parents=True, exist_ok=True)
subprocess.run(['rm', '-rf', '/content/data'])
Path('/content/data').symlink_to(DATA)
print('/content/data ->', Path('/content/data').resolve())

if not (REPO / 'scripts').is_dir():
    subprocess.run(['git', 'clone', '-q',
                    'https://github.com/Lion4467cat/dotav2-obb-benchmark.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'fetch', '-q', 'origin'], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '-q', COMMIT], check=True)
print('repo at', subprocess.run(['git', '-C', str(REPO), 'rev-parse', '--short', 'HEAD'],
                                capture_output=True, text=True).stdout.strip())
n_gpu = len(subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout.splitlines())
print('GPUs visible:', n_gpu)
assert n_gpu >= 1, 'no GPU: enable the T4 x2 accelerator'

# ---- discover the CLEAN retrain checkpoint mounted from the first kernel ----
INP = Path('/kaggle/input')
def parse_ep(p):
    m = re.search(r'epoch_(\d+)\.pth$', p.name)
    return int(m.group(1)) if m else -1

all_pth = sorted(INP.rglob('*.pth')) if INP.is_dir() else []
print('all .pth under /kaggle/input:')
for p in all_pth:
    print('   ', p, p.stat().st_size)
big = [p for p in all_pth if p.stat().st_size > 700e6]
plain = [p for p in big if re.match(r'epoch_\d+\.pth$', p.name) is not None]
bests = [p for p in big if p.name.startswith('best_')]
pool = plain if plain else bests
by_ep = {}
for p in pool:
    e = parse_ep(p)
    if e >= 0:
        by_ep.setdefault(e, p)
assert by_ep, 'no clean epoch checkpoint under /kaggle/input - is kernel_sources set?'
EPOCH = max(by_ep)
CKPT = by_ep[EPOCH]
print('chosen epoch', EPOCH, '->', CKPT, round(CKPT.stat().st_size / 1e6), 'MB')

def find_one(name):
    hits = sorted(INP.rglob(name)) if INP.is_dir() else []
    return hits[0] if hits else None
SRC_TRAIN_LOG = find_one('train_stdout.log')
SRC_TRAIN_CFG = find_one('train_cfg.py')
SRC_EP16_CFG  = find_one('ep16_cfg.py')
print('source train_log', SRC_TRAIN_LOG)
print('source train_cfg', SRC_TRAIN_CFG)
print('source ep16_cfg ', SRC_EP16_CFG)


In [ ]:
#@title 2 - Provision the Python 3.11 venv with the pinned OpenMMLab stack { display-mode: "form" }
import subprocess, time, os
from pathlib import Path

CHILD_ENV = dict(os.environ)
CHILD_ENV['MPLBACKEND'] = 'Agg'

if (VENV / 'bin' / 'python').is_file():
    print('venv already present at', VENV)
else:
    t0 = time.time()
    p = subprocess.Popen(['bash', str(REPO / 'scripts' / 'setup_colab.sh'), str(REPO), str(VENV)],
                         env=CHILD_ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='', flush=True)
    p.wait()
    print()
    print('setup exit', p.returncode, 'after', round((time.time() - t0) / 60, 1), 'min')
    assert p.returncode == 0, 'setup_colab.sh failed'
print('venv python', (VENV / 'bin' / 'python').resolve())


In [ ]:
#@title 3 - Verify the venv and apply the two mmrotate patches { display-mode: "form" }
import subprocess
from pathlib import Path

PY = VENV / 'bin' / 'python'

PROBE = '''
import torch, mmcv, mmengine, mmdet, os, mmrotate
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '|', torch.version.cuda)
print('device count', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print('  gpu', i, torch.cuda.get_device_name(i))
print('mmcv', mmcv.__version__, '| mmengine', mmengine.__version__, '| mmdet', mmdet.__version__)
print('mmrotate', os.path.dirname(mmrotate.__file__))
'''
r = subprocess.run([str(PY), '-c', PROBE], capture_output=True, text=True)
print(r.stdout)
assert r.returncode == 0, r.stderr[-2000:]

PATCH = '''
import sys
from pathlib import Path
sys.path.insert(0, 'REPO')
from scripts.run_kaggle import patch_gaussian_dist_loss, patch_rotate_iou2d_calculator
root = Path('REPO')
patch_gaussian_dist_loss(root)
patch_rotate_iou2d_calculator(root)
'''.replace('REPO', str(REPO))
r = subprocess.run([str(PY), '-c', PATCH], capture_output=True, text=True)
print(r.stdout.strip())
assert r.returncode == 0, r.stderr[-2000:]


In [ ]:
#@title 4 - Official DOTA v2.0 VAL tiling done HERE on Kaggle, then PROVE it { display-mode: "form" }
import subprocess, time, os, shutil as _sh
from collections import Counter
from pathlib import Path

SPLIT = DATA / 'split_ss_dotav2'
RAW = DATA / 'dotav2_raw'
CACHE = DATA / 'raw_archives'
PY = VENV / 'bin' / 'python'

def sh(cmd, cwd=None):
    print('+', ' '.join(str(c) for c in cmd), flush=True)
    p = subprocess.run([str(c) for c in cmd], cwd=str(cwd) if cwd else None)
    assert p.returncode == 0, 'command failed: ' + ' '.join(str(c) for c in cmd)

def _free(paths):
    for p in paths:
        try:
            if p.is_dir():
                _sh.rmtree(p, ignore_errors=True)
            elif p.is_file():
                p.unlink()
        except OSError:
            pass

t0 = time.time()
have = (SPLIT / 'val' / 'annfiles').is_dir() and any((SPLIT / 'val' / 'images').glob('*.png'))
if not have:
    # Proven official downloader: HF isaaccorley/dota (v2.0 annotations + v2.0-new images).
    sh([PY, REPO / 'scripts' / 'download_dotav2.py', '--source', 'huggingface',
        '--cache-dir', CACHE, '--out-dir', RAW], cwd=REPO)
    _free([RAW / '_extracted', *list(CACHE.rglob('*.tar.gz')), *list(CACHE.rglob('*.zip'))])
    _free([RAW / 'train'])   # only val is needed here

    # v2.0 image tarballs hold ONLY the 135 NEW val images; official val = 458 v1.0 + 135 = 593.
    url = 'https://huggingface.co/datasets/isaaccorley/dota/resolve/main/dotav1.0_images_val.tar.gz'
    (CACHE / 'val').mkdir(parents=True, exist_ok=True)
    arc = CACHE / 'val' / url.rsplit('/', 1)[-1]
    if not arc.is_file():
        sh(['curl', '-sL', '--retry', '3', '-o', str(arc), url])
    tmp = CACHE / '_v1_val'
    _sh.rmtree(tmp, ignore_errors=True)
    tmp.mkdir(parents=True)
    sh(['tar', 'xzf', str(arc), '-C', str(tmp)])
    imgdirs = [p for p in tmp.rglob('images') if p.is_dir()]
    assert imgdirs, 'no images dir in v1.0 val archive'
    best = max(imgdirs, key=lambda p: len(list(p.glob('*.png'))))
    (RAW / 'val' / 'images').mkdir(parents=True, exist_ok=True)
    n = 0
    for f in best.glob('*.png'):
        dst = RAW / 'val' / 'images' / f.name
        if not dst.exists():
            try:
                os.link(f, dst)
            except OSError:
                _sh.copy2(f, dst)
            n += 1
    print('v1.0 val images merged:', n)
    _free([tmp, arc])

    _driver = Path('/tmp/tile_one_split.py')
    _driver.write_text(
        'import sys, importlib.util\n'
        'from pathlib import Path\n'
        'spec = importlib.util.spec_from_file_location("_pd", sys.argv[1])\n'
        'pd = importlib.util.module_from_spec(spec); spec.loader.exec_module(pd)\n'
        'raw, splitdir, mmrotate, split = Path(sys.argv[2]), Path(sys.argv[3]), Path(sys.argv[4]), sys.argv[5]\n'
        'src = raw / split; save = splitdir / split\n'
        'import shutil as _sh\n'
        'if save.exists(): _sh.rmtree(save)\n'
        'cfg = pd.build_split_config([str(src / "images")], [str(src / "labelTxt")], save, 1024, 200, 4)\n'
        'pd.run_img_split(mmrotate, cfg)\n'
        'print("[tile]", split, "patches", len(list((save / "images").glob("*.png"))))\n'
    )
    sh([PY, _driver, REPO / 'scripts' / 'prepare_dotav2_patches.py', RAW, SPLIT,
        REPO / 'mmrotate', 'val'], cwd=REPO)
    _free([RAW / 'val' / 'images'])
print('val tiling ready in', round((time.time() - t0) / 60, 1), 'min')

def scan(ann_dir):
    classes, flags, inst = Counter(), Counter(), 0
    files = list(Path(ann_dir).glob('*.txt'))
    for f in files:
        for line in f.read_text(errors='ignore').splitlines():
            p = line.split()
            if len(p) < 10 or p[0].lower() in ('imagesource:', 'gsd:'):
                continue
            classes[p[8]] += 1
            flags[p[9]] += 1
            inst += 1
    return len(files), inst, classes, flags

V2_CLASSES = {'plane', 'ship', 'storage-tank', 'baseball-diamond', 'tennis-court',
              'basketball-court', 'ground-track-field', 'harbor', 'bridge', 'large-vehicle',
              'small-vehicle', 'helicopter', 'roundabout', 'soccer-ball-field',
              'swimming-pool', 'container-crane', 'airport', 'helipad'}

def ann_root(kind):
    direct = RAW / kind / 'labelTxt'
    if direct.is_dir():
        return direct
    hits = [p for p in RAW.rglob('version2.0') if p.is_dir() and kind in str(p)]
    return hits[0] if hits else None

src_val = ann_root('val')
assert src_val is not None, 'cannot locate the source val labelTxt'
va_f, va_i, va_c, va_fg = scan(src_val)
p_va_f, p_va_i, p_va_c, p_va_fg = scan(SPLIT / 'val' / 'annfiles')
n_va_img = len(list((SPLIT / 'val' / 'images').glob('*.png')))
print('SOURCE val', va_f, 'files /', va_i, 'instances /', len(va_c), 'classes')
print('PATCH  val', n_va_img, 'images /', p_va_f, 'annfiles /', p_va_i, 'instances')
print('PATCH  val non-difficult', p_va_fg.get('0', 0),
      '| difficult(diff!=0)', sum(v for k, v in p_va_fg.items() if k != '0'),
      '(flag1', p_va_fg.get('1', 0), 'flag2', p_va_fg.get('2', 0), ')')

checks = []
def check(name, ok, detail=''):
    checks.append((name, bool(ok)))
    print(('  PASS  ' if ok else '  FAIL  ') + name + ('   ' + str(detail) if detail else ''))

print()
# Source exact counts are informational; the authoritative, fatal proof is the
# PATCH-level checks below (7122 / 69542 / 70561 / 18 classes), proven in the first kernel.
print('  INFO   source val annfiles =', va_f, '(official 593)')
print('  INFO   source val instances =', va_i, '(official 81048)')
check('source has 18 classes', len(va_c) == 18, len(va_c))
check('airport present', 'airport' in va_c, va_c.get('airport', 0))
check('helipad present', 'helipad' in va_c, va_c.get('helipad', 0))
check('container-crane present', 'container-crane' in va_c, va_c.get('container-crane', 0))
check('difficulty flags present at source',
      va_fg.get('0', 0) > 0 and va_fg.get('1', 0) > 0,
      'flag0=' + str(va_fg.get('0', 0)) + ' flag1=' + str(va_fg.get('1', 0)))
check('val patches == 7122', n_va_img == 7122, n_va_img)
check('val annfiles == 7122', p_va_f == 7122, p_va_f)
check('val non-difficult instances == 69542', p_va_fg.get('0', 0) == 69542, p_va_fg.get('0', 0))
check('val difficult instances == 70561 (diff!=0)',
      sum(v for k, v in p_va_fg.items() if k != '0') == 70561,
      'flag1=' + str(p_va_fg.get('1', 0)) + ' flag2=' + str(p_va_fg.get('2', 0)))
check('18 classes at patch level', len(p_va_c) == 18, len(p_va_c))

failed = [n for n, ok in checks if not ok]
assert not failed, 'VAL VERIFICATION FAILED: ' + ', '.join(failed)
print()
print('val verification: all', len(checks), 'checks passed')


In [ ]:
#@title 5 - Epoch-22 official validation (dota/mAP) + final extended eval { display-mode: "form" }
import subprocess, json, os, time, re, shutil
from pathlib import Path

PY = VENV / 'bin' / 'python'

# Config straight out of the checkpoint (guarantees architecture match). Read with the venv python.
CFG_META = WORK / 'ckpt_meta_cfg.py'
code = (
    "import torch, pathlib\n"
    "ck = torch.load(" + repr(str(CKPT)) + ", map_location='cpu')\n"
    "cfg = ck['meta']['cfg']\n"
    "cfg = cfg if isinstance(cfg, str) else getattr(cfg, 'pretty_text', str(cfg))\n"
    "pathlib.Path(" + repr(str(CFG_META)) + ").write_text(cfg)\n"
    "print('config extracted, chars', len(cfg))\n"
)
r = subprocess.run([str(PY), '-c', code], capture_output=True, text=True)
print(r.stdout.strip())
assert r.returncode == 0, r.stderr[-2000:]
cfg_full = CFG_META.read_text()
assert 'GDLoss' in cfg_full and 'ConvNeXt' in cfg_full, 'checkpoint config is not Track B'
print('ckpt cfg work_dir lines:', [l.strip() for l in cfg_full.splitlines() if 'work_dir' in l][:3])

# Strip the appended Kaggle TRAIN overrides; keep the clean base (== ep16_cfg.py), then eval-only.
marker = '# ---- official-retrain overrides'
cfg_base = cfg_full.split(marker)[0].rstrip() + '\n' if marker in cfg_full else cfg_full
(WORK / 'ep16_cfg.py').write_text(cfg_base)

EVAL_CFG = WORK / 'eval_cfg.py'
extra = '''
# ---- eval-only overrides (recipe above untouched) ----
resume = False
load_from = '%s'
work_dir = '%s'
custom_hooks = []
''' % (CKPT, WORK / 'eval_test')
EVAL_CFG.write_text(cfg_base + extra)
print('eval config ->', EVAL_CFG)

# Recover honest peak VRAM from the first kernel's training log (scanned by run_track_eval.py).
if SRC_TRAIN_LOG and SRC_TRAIN_LOG.is_file():
    shutil.copy2(SRC_TRAIN_LOG, WORK / 'train_stdout.log')

env = dict(os.environ)
env['MPLBACKEND'] = 'Agg'
env['PYTHONPATH'] = str(REPO) + ':' + str(REPO / 'mmrotate') + ':' + env.get('PYTHONPATH', '')
env['PYTORCH_CUDA_ALLOC_CONF'] = 'max_split_size_mb:128'

cmd = [str(PY), str(REPO / 'scripts' / 'run_track_eval.py'),
       '--config', str(EVAL_CFG),
       '--work-dir', str(WORK),
       '--checkpoint', str(CKPT),
       '--ann-dir', str(SPLIT / 'val' / 'annfiles'),
       '--batch-size', '1']
print('+', ' '.join(cmd))
t0 = time.time()
p = subprocess.Popen(cmd, cwd=str(REPO), env=env, stdout=subprocess.PIPE,
                     stderr=subprocess.STDOUT, text=True, bufsize=1)
lines = []
for line in p.stdout:
    print(line, end='', flush=True)
    lines.append(line)
p.wait()
out = ''.join(lines)
(FIN / 'logs' / 'eval_stdout.log').write_text(out)
assert p.returncode == 0, 'evaluation failed (rc=%d)' % p.returncode

# EPOCH-22 official validation: test_evaluator == val_evaluator == DOTAMetric('mAP').
map_m = re.findall(r'dota/mAP[:\s|]*([0-9]*\.?[0-9]+)', out)
ap50_m = re.findall(r'dota/AP50[:\s|]*([0-9]*\.?[0-9]+)', out)
dota_map = float(map_m[-1]) if map_m else None
dota_ap50 = float(ap50_m[-1]) if ap50_m else None
print('EPOCH %d official validation: dota/mAP=%s  dota/AP50=%s' % (EPOCH, dota_map, dota_ap50))
if dota_map is None:
    print('WARNING: could not parse dota/mAP from eval log; will be marked null in the JSON.')

EXT = WORK / 'eval_test' / 'extended_metrics.json'
assert EXT.is_file(), 'missing ' + str(EXT)
metrics = json.loads(EXT.read_text())
for k in ('num_images', 'num_gt_non_difficult', 'num_predictions', 'map50', 'ap75', 'map50_95',
          'class_agnostic_ap50', 'classification_gap_ap50'):
    print('  ', k, '=', metrics[k])
print('   operational =', metrics.get('operational'))
print('   difficulty_recall =', metrics.get('difficulty_recall'))
print('elapsed', round((time.time() - t0) / 60, 1), 'min')


In [ ]:
#@title 6 - Honest comparison table, results JSON, deviations (epoch 22 / 17-22) { display-mode: "form" }
import json, shutil, re
from pathlib import Path

metrics = json.loads((WORK / 'eval_test' / 'extended_metrics.json').read_text())

BASELINE = {'classes': ['plane', 'baseball-diamond', 'bridge', 'ground-track-field', 'small-vehicle', 'large-vehicle', 'ship', 'tennis-court', 'basketball-court', 'storage-tank', 'soccer-ball-field', 'roundabout', 'harbor', 'swimming-pool', 'helicopter', 'container-crane', 'airport', 'helipad'], 'track_a_ep10': {'map50': 0.6049, 'ap75': 0.2785, 'map50_95': 0.3146, 'class_agnostic_ap50': 0.7985, 'operational': {'fps': 11.36, 'params_m': None, 'peak_vram_gb': 2.86}, 'per_class': {'plane': {'ap50': 0.9072}, 'baseball-diamond': {'ap50': 0.6984}, 'bridge': {'ap50': 0.3901}, 'ground-track-field': {'ap50': 0.7285}, 'small-vehicle': {'ap50': 0.7653}, 'large-vehicle': {'ap50': 0.7296}, 'ship': {'ap50': 0.8237}, 'tennis-court': {'ap50': 0.9385}, 'basketball-court': {'ap50': 0.6067}, 'storage-tank': {'ap50': 0.8081}, 'soccer-ball-field': {'ap50': 0.5525}, 'roundabout': {'ap50': 0.7123}, 'harbor': {'ap50': 0.6067}, 'swimming-pool': {'ap50': 0.6367}, 'helicopter': {'ap50': 0.5346}, 'container-crane': {'ap50': 0.0}, 'airport': {'ap50': 0.4492}, 'helipad': {'ap50': 0.0}}}, 'track_b_old': {'map50': 0.6968, 'ap75': 0.4347, 'map50_95': 0.4175, 'class_agnostic_ap50': 0.8765, 'operational': {'fps': 5.7, 'params_m': 70.24, 'peak_vram_gb': 4.2}, 'per_class': {'plane': {'ap50': 0.9359}, 'baseball-diamond': {'ap50': 0.7567}, 'bridge': {'ap50': 0.5573}, 'ground-track-field': {'ap50': 0.7983}, 'small-vehicle': {'ap50': 0.822}, 'large-vehicle': {'ap50': 0.8687}, 'ship': {'ap50': 0.9111}, 'tennis-court': {'ap50': 0.957}, 'basketball-court': {'ap50': 0.7621}, 'storage-tank': {'ap50': 0.8286}, 'soccer-ball-field': {'ap50': 0.6723}, 'roundabout': {'ap50': 0.7628}, 'harbor': {'ap50': 0.8027}, 'swimming-pool': {'ap50': 0.6686}, 'helicopter': {'ap50': 0.6685}, 'container-crane': {'ap50': 0.0323}, 'airport': {'ap50': 0.4876}, 'helipad': {'ap50': 0.25}}}}

A = BASELINE['track_a_ep10']
B_OLD = BASELINE['track_b_old']

def f2(x):
    return ('%.2f' % x) if isinstance(x, (int, float)) else 'n/a'

NEW_LABEL = 'Track B NEW ep%d (official retrain)' % EPOCH
lines = ['| Metric | Track A ep10 (R50) | Track B OLD ep24 (official re-score) | ' + NEW_LABEL + ' | delta NEW-OLD |',
         '|---|---|---|---|---|']
for label, key in (('mAP50', 'map50'), ('AP75', 'ap75'), ('mAP50-95', 'map50_95'),
                   ('class-agnostic AP50', 'class_agnostic_ap50')):
    a, b, n = A[key], B_OLD[key], metrics[key]
    lines.append('| %s | %.4f | %.4f | %.4f | %+.4f |' % (label, a, b, n, n - b))
op = metrics.get('operational', {}) or {}
pm = op.get('params_m')
pm_delta = ('%+.2f' % (pm - B_OLD['operational']['params_m'])) if isinstance(pm, (int, float)) else 'n/a'
lines.append('| params (M) | %s | %.2f | %s | %s |' % (
    f2(A['operational'].get('params_m')), B_OLD['operational']['params_m'], f2(pm), pm_delta))
table = '\n'.join(lines)

per = ['| Class | Track B OLD AP50 | Track B NEW AP50 | delta |', '|---|---|---|---|']
for cls in BASELINE['classes']:
    if cls in metrics['per_class'] and cls in B_OLD['per_class']:
        o = B_OLD['per_class'][cls]['ap50']
        n = metrics['per_class'][cls]['ap50']
        per.append('| %s | %.4f | %.4f | %+.4f |' % (cls, o, n, n - o))
per_table = '\n'.join(per)

comparison = ('# Track B official retrain - comparison' + '\n\n' + table + '\n\n'
              + '## Per-class AP50' + '\n\n' + per_table + '\n\n'
              + '> Scope: Track B NEW = epoch_%d (last clean checkpoint). Epochs 23 and 24 were\n'
              + '> intentionally skipped; the 24-epoch 2x schedule is NOT complete.' % EPOCH + '\n')
(FIN / 'comparison_table.md').write_text(comparison)

per_class = metrics['per_class']
assert len(per_class) == 18, len(per_class)
mean_ap50 = sum(v['ap50'] for v in per_class.values()) / 18
mean_ap = sum(v['ap50_95'] for v in per_class.values()) / 18
print('mean per-class AP50 %.4f vs headline %.4f' % (mean_ap50, metrics['map50']))
print('mean per-class AP50-95 %.4f vs headline %.4f' % (mean_ap, metrics['map50_95']))
assert abs(mean_ap50 - metrics['map50']) < 5e-4
assert abs(mean_ap - metrics['map50_95']) < 5e-4
assert metrics['num_gt_non_difficult'] == 69542, 'val tiling is not the official 7,122-patch set'

# Recover the in-training validation history from the first kernel's training log.
val_hist = {}
if SRC_TRAIN_LOG and SRC_TRAIN_LOG.is_file():
    txt = SRC_TRAIN_LOG.read_text(errors='ignore')
    for m in re.finditer(r'Epoch\(val\) \[(\d+)\]\[\d+/\d+\]\s+dota/mAP:\s*([0-9.]+)\s+dota/AP50:\s*([0-9.]+)', txt):
        val_hist[int(m.group(1))] = {'dota/mAP': float(m.group(2)), 'dota/AP50': float(m.group(3))}
val_hist[EPOCH] = {'dota/mAP': dota_map, 'dota/AP50': dota_ap50,
                   'note': 'completed in this finishing notebook (DOTAMetric; test_evaluator == val_evaluator)'}
print('validation history:', val_hist)

results = {
    'model': 'oriented_rcnn_convnext_s_pafpn_gaussian_kld',
    'checkpoint': CKPT.name,
    'final_epoch': EPOCH,
    'trained_epochs': '17-22 on official DOTA v2.0 trainval (18 classes, difficulty flags preserved); resumed from epoch_16.pth',
    'schedule_status': 'BUDGET-LIMITED: epochs 23 and 24 intentionally SKIPPED. The retrain kernel hit the Kaggle 12h cap during the epoch-22 validation. This is NOT a completed 24-epoch schedule.',
    'validation': val_hist,
    'epoch_22_validation': {'dota/mAP': dota_map, 'dota/AP50': dota_ap50},
    'gpu': '2 x T4 16GB (Kaggle)',
    'effective_batch': 8,
    'official_val': {'val_images': 593, 'val_patches': 7122, 'source_instances': 81048,
                     'gt_non_difficult': 69542, 'gt_difficult': 70561, 'classes': 18},
    'metrics': {k: metrics[k] for k in ('num_images', 'num_gt_non_difficult', 'num_predictions',
                                        'map50', 'ap75', 'map50_95', 'class_agnostic_ap50',
                                        'classification_gap_ap50', 'micro_prf', 'macro_f1_opt',
                                        'scale_ap50', 'orientation_error_deg', 'difficulty_recall',
                                        'operational') if k in metrics},
    'per_class': per_class,
    'baseline_track_b_old': {k: B_OLD[k] for k in ('map50', 'ap75', 'map50_95', 'class_agnostic_ap50')},
    'baseline_track_a_ep10': {k: A[k] for k in ('map50', 'ap75', 'map50_95', 'class_agnostic_ap50')},
}
(FIN / 'results_trackB_official_retrain.json').write_text(json.dumps(results, indent=2))

deviations = [
    '# Deviations and honest scope (finishing notebook)',
    '',
    'Every entry below is environment/decision-forced. Model, loss, optimizer, LR schedule,',
    'augmentation, sampler and seed policy are unchanged from the original recipe.',
    '',
    '1. Hardware: 2 x T4 16GB (Kaggle) instead of 1 x A100 then 2 x T4. Effective batch kept at 8',
    '   by 1 sample per GPU with 4-step gradient accumulation.',
    '2. Runtime: a Python 3.11 venv (scripts/setup_colab.sh) carries the pinned',
    '   torch 2.0.1 / mmcv 2.0.1 / mmdet 3.1.0 wheels without a source build.',
    '3. work_dir is new; the Phase-1 run and the old contaminated checkpoints are never touched.',
    '4. Epochs 17-22 were trained cleanly on the official full DOTA v2.0 train/val split',
    '   (1830 train / 593 val images, 18 classes, difficulty flags preserved), resumed at epoch 16.',
    '5. EPOCHS 23 AND 24 ARE INTENTIONALLY SKIPPED. The 2x (24-epoch) schedule is therefore NOT',
    '   complete: this is a budget-limited 6-epoch official continuation, not a converged 24-epoch run.',
    '6. The retrain kernel hit the Kaggle 12h cap during the epoch-22 validation. This finishing',
    '   notebook re-tiled the official 7,122-patch val split HERE on Kaggle and completed that',
    '   validation plus the final extended evaluation on the last clean checkpoint (epoch_%d.pth).' % EPOCH,
    '7. Validation cadence val_interval=2 -> epochs 18, 20, 22 validated; epoch 24 not reached.',
    '8. The chosen checkpoint is copied into this notebook\'s /kaggle/working output so the',
    '   deliverable is self-contained on Kaggle (not dependent on the first kernel\'s output).',
    '9. The two mmrotate source patches used by the original run are re-applied: analytical 2x2',
    '   determinant in the Gaussian KLD loss, and chunked rotated-IoU overlaps.',
    '',
    'Not changed: model architecture, Gaussian KLD loss, AdamW base LR 1e-4, LinearLR warmup 500',
    'iters, CosineAnnealingLR T_max 24 eta_min 1e-6, ClassBalancedDataset oversample_thr 1e-3,',
    'all augmentation pipelines, seed 1143503445.',
]
(FIN / 'deviations.md').write_text('\n'.join(deviations))

for src, name in ((SRC_TRAIN_LOG, 'train_stdout.log'),
                  (SRC_TRAIN_CFG, 'train_cfg.py'),
                  (SRC_EP16_CFG, 'ep16_cfg.py'),
                  (CFG_META, 'ckpt_meta_cfg.py'),
                  (EVAL_CFG, 'eval_cfg.py')):
    if src and Path(src).is_file():
        shutil.copy2(src, FIN / name)

print()
print(table)
print()
print('wrote deliverables to', FIN)


In [ ]:
#@title 7 - Save ALL important files to /kaggle/working (Kaggle output) + manifest { display-mode: "form" }
import shutil, json, hashlib
from pathlib import Path
from datetime import datetime, timezone

def sha256(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for c in iter(lambda: f.read(1 << 20), b''):
            h.update(c)
    return h.hexdigest()

def cp(src, dst):
    if src and Path(src).is_file():
        Path(dst).parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
        print('  +', Path(dst).relative_to(FIN))

# Durable, self-contained copy of the clean checkpoint.
cp(CKPT, FIN / 'checkpoints' / CKPT.name)
# Eval outputs.
cp(WORK / 'eval_test' / 'extended_metrics.json', FIN / 'extended_metrics.json')
cp(WORK / 'eval_test' / 'preds.pkl', FIN / 'preds.pkl')
# Val annfiles (small; makes the tiling reproducible without the 8.6GB patch PNGs).
va_src = SPLIT / 'val' / 'annfiles'
if va_src.is_dir():
    (FIN / 'val_annfiles').mkdir(parents=True, exist_ok=True)
    n = 0
    for f in va_src.glob('*.txt'):
        shutil.copy2(f, FIN / 'val_annfiles' / f.name)
        n += 1
    print('  + val_annfiles:', n, 'txt')
(FIN / 'tiling_recipe.json').write_text(json.dumps({
    'source': 'huggingface isaaccorley/dota (v2.0 annotations + v2.0-new images) + v1.0 images',
    'val_images': 593, 'val_source_instances': 81048,
    'patch_size': 1024, 'gap': 200, 'img_rate_thr': 0.6, 'iof_thr': 0.7,
    'no_padding': False, 'padding_value': [104, 116, 124],
    'val_patches': 7122,
    'note': 'patch PNGs (~8.6GB) intentionally NOT stored; re-tile with scripts/prepare_dotav2_patches.py',
}, indent=2))

files = []
for p in sorted(FIN.rglob('*')):
    if p.is_file() and p.name != 'manifest.json':
        files.append({'path': str(p.relative_to(FIN)), 'size': p.stat().st_size,
                      'sha256': sha256(p)})
manifest = {
    'artifact': 'trackB official retrain - FINISHING deliverables',
    'captured_at_utc': datetime.now(timezone.utc).isoformat(),
    'source_kernel': SRC_SLUG,
    'final_checkpoint': CKPT.name,
    'final_epoch': EPOCH,
    'epochs_23_24': 'SKIPPED (intentionally; Kaggle 12h cap)',
    'epoch_22_validation': {'dota/mAP': dota_map, 'dota/AP50': dota_ap50},
    'official_val_patches': 7122,
    'n_files': len(files),
    'files': files,
}
(FIN / 'manifest.json').write_text(json.dumps(manifest, indent=2))

print()
print('=== /kaggle/working/ada_trackB_official_finish ===')
for p in sorted(FIN.rglob('*')):
    if p.is_file():
        print('  %12d  %s' % (p.stat().st_size, p.relative_to(FIN)))
print()
print('SUMMARY_JSON', json.dumps({
    'epoch': EPOCH, 'checkpoint': CKPT.name,
    'dota/mAP': dota_map, 'dota/AP50': dota_ap50,
    'map50': metrics['map50'], 'ap75': metrics['ap75'], 'map50_95': metrics['map50_95'],
    'old_map50': B_OLD['map50'], 'old_map50_95': B_OLD['map50_95'],
    'n_files': len(files)}))
